In [0]:
fact_appointments = spark.sql(f"select * from regis_healthcare.silver.appointments;")
fact_appointments.createOrReplaceTempView("appointments")

In [0]:
# Fact_Appointments -- > Source: appointments
# | Foreign Keys         |
# | -------------------- |
# | appointment_key      |
# | resident_key         |
# | employee_key         |
# | appointment_type_key |
# | appointment_date_key |
# --$
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

fact_appointments = fact_appointments.select(
    "appointment_type"
).distinct()
window_spec = Window.orderBy("appointment_type")

fact_appointments = fact_appointments.withColumn(
    "appointment_type_key",
    row_number().over(window_spec)
)
fact_appointments = fact_appointments.select(
"appointment_type_key",
"appointment_type")
fact_appointments.createOrReplaceTempView("appointmentss")
fact_appointments =spark.sql("""
 select a.*,s.appointment_type_key from appointments as a join appointmentss as s 
 on a.appointment_type = s.appointment_type
""")
# #--$
from pyspark.sql.functions import col, date_format
# Create date_key column in YYYYMMDD format
fact_appointments = fact_appointments.withColumn("appointment_date_key", date_format(col("appointment_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
fact_appointments = fact_appointments.withColumn("appointment_date_key", col("appointment_date_key").cast("int"))
#-------------------------
from pyspark.sql.functions import col, regexp_replace
fact_appointments = fact_appointments.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)
fact_appointments = fact_appointments.withColumn(
    "appointment_key",
    regexp_replace(col("appointment_id"), "^APT", "").cast("int")
)

fact_appointments = fact_appointments.withColumn(
    "employee_key",
    regexp_replace(col("employee_id"), "^EMP", "").cast("int")
)

fact_appointments = fact_appointments.select(
"appointment_key",     
 "resident_key",      
 "employee_key",     
 "appointment_type_key", 
 "appointment_date_key" 
)

display(fact_appointments)



#### cataloge 

In [0]:
fact_appointments.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.fact_appointments")

In [0]:
fact_appointments.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_fact_appointments")
print(fact_appointments.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.fact_appointments")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("regis_healthcare.gold.sb_fact_appointments")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.appointment_key = source.appointment_key"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.fact_appointments;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_fact_appointments;")
print(sb_dim_df.count())

#### s3 loading

In [0]:
# gold load to s3
fact_appointments.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.save(f"s3://regis-healthcare/gold-delta-table/fact_appointments")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/fact_appointments"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = fact_appointments

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.appointment_key = source.appointment_key"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)
